# Vehicle speed detection

STEP 1  : Import required libraries

STEP 2  : Define user settings (video, lines, real distance)

STEP 3  : Define colors for visualization

STEP 4  : Load YOLO model with built-in tracking

STEP 5  : Initialize video capture and FPS

STEP 6  : Initialize tracking data structures

STEP 7  : Read video frame-by-frame

STEP 8  : Perform YOLO detection + tracking

STEP 9  : Draw reference lines (Line A & Line B)

STEP 10 : Filter vehicle classes only

STEP 11 : Calculate object center point

STEP 12 : Detect line crossing events

STEP 13 : Store frame numbers for crossings

STEP 14 : Compute speed using distance & time

STEP 15 : Draw bounding boxes, IDs, speed labels

STEP 16 : Display output frame

STEP 17 : Exit and cleanup


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

# ----------------------------
# SETTINGS
# ----------------------------
VIDEO_PATH = "/content/drive/MyDrive/YOLO/highway.mp4"

# ----------------------------
# MODEL
# ----------------------------
model = YOLO("/content/drive/MyDrive/YOLO/yolo26n.pt")

# ----------------------------
# VIDEO
# ----------------------------
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    fps = 30

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Dynamically set line Y positions relative to video height (40% and 45% down the frame)
LINE_Y1 = int(height * 0.40)
LINE_Y2 = int(height * 0.45)
REAL_DISTANCE_M = 10  # metres between the two lines

# Dynamically calculate road boundaries based on actual video width
mid_x = width // 2

# --- Road A (LEFT) ---
road_A = {
    "name": "Road A",
    "first_line":  [(0, LINE_Y1),  (mid_x, LINE_Y1)],
    "second_line": [(0, LINE_Y2),  (mid_x, LINE_Y2)],
    "color_line1": (0,   255, 255),   # cyan
    "color_line2": (255, 0,   0),     # blue
    "x_min": 0,
    "x_max": mid_x,
}

# --- Road B (RIGHT) ---
road_B = {
    "name": "Road B",
    "first_line":  [(mid_x, LINE_Y1), (width, LINE_Y1)],
    "second_line": [(mid_x, LINE_Y2), (width, LINE_Y2)],
    "color_line1": (0,   255, 255),   # cyan
    "color_line2": (255, 0,   0),     # blue
    "x_min": mid_x,
    "x_max": width,
}

ROADS = [road_A, road_B]

out = cv2.VideoWriter(
    "output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

# ----------------------------
# PER-TRACK STATE
# ----------------------------
cross_data = {}
frame_id = 0

# ----------------------------
# HELPER – assign road by cx
# ----------------------------
def get_road(cx):
    for road in ROADS:
        if road["x_min"] <= cx < road["x_max"]:
            return road
    return None

# ----------------------------
# COUNTERS (per road)
# ----------------------------
count_down = {r["name"]: [] for r in ROADS}
count_up   = {r["name"]: [] for r in ROADS}

# ----------------------------
# MAIN LOOP
# ----------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1
    current_time = frame_id / fps  # Actual video timestamp (fixes time.time() bug)

    # YOLO tracking
    results = model.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False)[0]

    # Draw lines for BOTH roads
    for road in ROADS:
        cv2.line(frame, road["first_line"][0],  road["first_line"][1],  road["color_line1"], 3)
        cv2.line(frame, road["second_line"][0], road["second_line"][1], road["color_line2"], 3)

    # Road label overlays
    cv2.putText(frame, "Road A", (20, LINE_Y1 - 20),  cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
    cv2.putText(frame, "Road B", (mid_x + 20, LINE_Y1 - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)

    if results.boxes is not None:
        for box in results.boxes:
            cls = int(box.cls[0])
            if cls not in [2, 3, 5, 7]:   # car, motorcycle, bus, truck
                continue

            if box.id is None:
                continue

            track_id = int(box.id[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

            # Determine which road this vehicle is on
            road = get_road(cx)
            if road is None:
                continue

            road_name = road["name"]

            # Initialise state for new track IDs
            if track_id not in cross_data:
                cross_data[track_id] = {
                    "road":  road_name,
                    "line1": None,
                    "line2": None,
                    "speed": None,
                    "prev_cy": cy, # Store previous Y to detect crossing
                }

            data = cross_data[track_id]
            prev_cy = data["prev_cy"]

            # -----------------------------------------------
            # ROBUST LINE CROSSING DETECTION
            # Checks if the vehicle crossed the line between the previous frame and now
            # -----------------------------------------------
            crossed_line1 = (prev_cy < LINE_Y1 and cy >= LINE_Y1) or (prev_cy > LINE_Y1 and cy <= LINE_Y1)
            crossed_line2 = (prev_cy < LINE_Y2 and cy >= LINE_Y2) or (prev_cy > LINE_Y2 and cy <= LINE_Y2)

            if crossed_line1 and data["line1"] is None:
                data["line1"] = current_time

            if crossed_line2 and data["line2"] is None:
                data["line2"] = current_time

            # Update previous cy for the next frame
            data["prev_cy"] = cy

            # -----------------------------------------------
            # SPEED & DIRECTION CALCULATION
            # -----------------------------------------------
            if data["speed"] is None and data["line1"] is not None and data["line2"] is not None:
                t = abs(data["line2"] - data["line1"])
                if t > 0:
                    speed = (REAL_DISTANCE_M / t) * 3.6
                    data["speed"] = speed

                    # Determine direction based on which line was crossed first
                    if data["line1"] < data["line2"]:
                        # Crossed line1 first -> Moving DOWN (Y increases)
                        if track_id not in count_down[road_name]:
                            count_down[road_name].append(track_id)
                    else:
                        # Crossed line2 first -> Moving UP (Y decreases)
                        if track_id not in count_up[road_name]:
                            count_up[road_name].append(track_id)

            # -----------------------------------------------
            # DRAW bounding box + label
            # -----------------------------------------------
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            if data["speed"] is not None:
                label = f"{data['speed']:.1f} km/h"
            else:
                label = f"ID {track_id}"

            cv2.putText(frame, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            cv2.circle(frame, (cx, cy), 5, (255, 255, 0), -1)

    # -----------------------------------------------
    # HUD – vehicle counts per road
    # -----------------------------------------------
    hud_y = 30
    for road in ROADS:
        rn = road["name"]
        hud_x = 20 if rn == "Road A" else mid_x + 20
        total = len(set(count_down[rn]) | set(count_up[rn]))
        cv2.putText(frame,
                    f"{rn}  |  Vehicles: {total}  (Down: {len(count_down[rn])}, Up: {len(count_up[rn])})",
                    (hud_x, hud_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)

    out.write(frame)

# ----------------------------
# CLEANUP
# ----------------------------
cap.release()
out.release()

print("✅ DONE! output.mp4 saved")
print(f"  Road A  ↓ {len(count_down['Road A'])}  ↑ {len(count_up['Road A'])}")
print(f"  Road B  ↓ {len(count_down['Road B'])}  ↑ {len(count_up['Road B'])}")

✅ DONE! output.mp4 saved
  Road A  ↓ 35  ↑ 0
  Road B  ↓ 0  ↑ 34
